# Notebook 34: GEE Data Acquisition — New Cities

Downloads GSHTD daily Tmin (2003–2020) for 14 new cities via parallel GEE batch exports
to Google Drive, then downloads and converts to NetCDF.

Uses the correct `global-daily-air-temp/{region}` daily collections (same as existing cities),
NOT `GSHTD/TMIN` which is monthly. Raw `b1` values are 10× °C; `divide(10)` gives °C.

## Workflow
1. Authenticate & initialise GEE
2. Config
3. UCDB match → bounding boxes
4. Submit GEE export tasks (all cities in parallel)
5. Monitor task status
6. Download from Drive & convert to NetCDF
7. Validate

In [1]:
import ee
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

## Step 1 — Authenticate & Initialise GEE

Run `ee.Authenticate()` once to store credentials. Subsequent runs use `ee.Initialize()` only.
If running for the first time, uncomment the `ee.Authenticate()` line.

In [ ]:
# ee.Authenticate()  # run once to store credentials
ee.Initialize(project='tl-cities')
print("GEE initialised")

## Step 2 — Config

In [ ]:
PIPELINE_DATA = Path('../heat_threshold_analysis/data')
UCDB_PATH     = PIPELINE_DATA / 'GHS_UCDB_2015_R2019A.gpkg'
START_DATE    = '2003-01-01'
END_DATE      = '2021-01-01'
BUFFER_DEG    = 0.05
GDRIVE_FOLDER = 'GSHTD_exports_v2'   # new folder to avoid confusion with wrong previous exports

# Correct daily collections — same series as existing cities.
# Filter prop_type='tmin', select band b1, divide(10) → °C.
AFRICA      = 'projects/sat-io/open-datasets/global-daily-air-temp/africa'
EUROPE_ASIA = 'projects/sat-io/open-datasets/global-daily-air-temp/europe_asia'

NEW_CITIES = {
    'accra':        {'label': 'Accra',        'country': 'Ghana',        'ucdb_aliases': ['Accra'],                                   'collection': AFRICA},
    'lagos':        {'label': 'Lagos',        'country': 'Nigeria',      'ucdb_aliases': ['Lagos'],                                   'collection': AFRICA},
    'ekurhuleni':   {'label': 'Ekurhuleni',   'country': 'South Africa', 'ucdb_aliases': ['Ekurhuleni', 'East Rand', 'Johannesburg'], 'collection': AFRICA},
    'kuala_lumpur': {'label': 'Kuala Lumpur', 'country': 'Malaysia',     'ucdb_aliases': ['Kuala Lumpur', 'Kuala lumpur'],            'collection': EUROPE_ASIA},
    'karachi':      {'label': 'Karachi',      'country': 'Pakistan',     'ucdb_aliases': ['Karachi'],                                 'collection': EUROPE_ASIA},
    'krishnagiri':  {'label': 'Krishnagiri',  'country': 'India',        'ucdb_aliases': ['Krishnagiri'],                            'collection': EUROPE_ASIA},
    'chennai':      {'label': 'Chennai',      'country': 'India',        'ucdb_aliases': ['Chennai', 'Madras'],                      'collection': EUROPE_ASIA},
    'kalburgi':     {'label': 'Kalburgi',     'country': 'India',        'ucdb_aliases': ['Kalaburagi', 'Gulbarga', 'Kalburgi'],     'collection': EUROPE_ASIA},
    'bidar':        {'label': 'Bidar',        'country': 'India',        'ucdb_aliases': ['Bidar'],                                  'collection': EUROPE_ASIA},
    'vijaypura':    {'label': 'Vijaypura',    'country': 'India',        'ucdb_aliases': ['Vijayapura', 'Bijapur', 'Vijaypura'],     'collection': EUROPE_ASIA},
    'raichur':      {'label': 'Raichur',      'country': 'India',        'ucdb_aliases': ['Raichur'],                                'collection': EUROPE_ASIA},
    'belgavi':      {'label': 'Belgavi',      'country': 'India',        'ucdb_aliases': ['Belagavi', 'Belgaum', 'Belgavi'],         'collection': EUROPE_ASIA},
    'ballari':      {'label': 'Ballari',      'country': 'India',        'ucdb_aliases': ['Ballari', 'Bellary'],                     'collection': EUROPE_ASIA},
    'mangaluru':    {'label': 'Mangaluru',    'country': 'India',        'ucdb_aliases': ['Mangaluru', 'Mangalore'],                 'collection': EUROPE_ASIA},
}

## Step 3 — Match Cities in UCDB & Extract Bounding Boxes

In [ ]:
ucdb = gpd.read_file(UCDB_PATH)
print(f"UCDB loaded: {len(ucdb)} urban centres")

def match_ucdb(ucdb, aliases, country):
    country_mask = ucdb['CTR_MN_NM'].str.lower().str.contains(country.lower(), na=False)
    candidates = ucdb[country_mask]
    for alias in aliases:
        name_mask = (
            (candidates['UC_NM_MN'].str.lower() == alias.lower()) |
            (candidates['UC_NM_LST'].str.lower().str.contains(alias.lower(), na=False))
        )
        hits = candidates[name_mask]
        if len(hits) > 0:
            return hits.loc[hits['P15'].idxmax()]
    return None

city_bounds = {}
rows = []
for slug, cfg in NEW_CITIES.items():
    match = match_ucdb(ucdb, cfg['ucdb_aliases'], cfg['country'])
    if match is not None:
        bounds = match.geometry.bounds
        city_bounds[slug] = {
            'label':      cfg['label'],
            'collection': cfg['collection'],
            'lon_min':    bounds[0] - BUFFER_DEG,
            'lat_min':    bounds[1] - BUFFER_DEG,
            'lon_max':    bounds[2] + BUFFER_DEG,
            'lat_max':    bounds[3] + BUFFER_DEG,
        }
        rows.append({'city': cfg['label'], 'matched': 'Yes', 'collection': cfg['collection'].split('/')[-1],
                     'bbox': f"{bounds[0]:.2f},{bounds[1]:.2f},{bounds[2]:.2f},{bounds[3]:.2f}"})
    else:
        rows.append({'city': cfg['label'], 'matched': 'NO MATCH', 'collection': '—', 'bbox': '—'})

print(pd.DataFrame(rows).to_string(index=False))
print(f"\n{len(city_bounds)} cities matched")

## Step 4 — Submit GEE Export Tasks (all cities in parallel)

In [ ]:
YEARS = list(range(2003, 2021))  # one task per city-year = ~365 bands each

tasks = {}  # keyed by (slug, year)

for slug, bounds in city_bounds.items():
    out_path = PIPELINE_DATA / f'{slug}_tmin.nc'
    if out_path.exists():
        print(f"  [{bounds['label']}] NetCDF already exists — skipping")
        continue

    roi = ee.Geometry.Rectangle([bounds['lon_min'], bounds['lat_min'],
                                  bounds['lon_max'], bounds['lat_max']])

    for year in YEARS:
        collection = (ee.ImageCollection(bounds['collection'])
                      .filter(ee.Filter.eq('prop_type', 'tmin'))
                      .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
                      .filterBounds(roi))

        def prep_image(img):
            date_str = img.date().format('YYYYMMdd')
            return (img.select('b1').divide(10).rename(date_str)
                       .set('system:time_start', img.get('system:time_start')))

        task = ee.batch.Export.image.toDrive(
            image=collection.map(prep_image).toBands(),
            description=f'{slug}_tmin_{year}',
            folder=GDRIVE_FOLDER,
            fileNamePrefix=f'{slug}_tmin_{year}',
            region=roi,
            scale=1000,
            crs='EPSG:4326',
            maxPixels=int(1e13),
            fileFormat='GeoTIFF',
        )
        task.start()
        tasks[(slug, year)] = task

    print(f"  [{bounds['label']}] {len(YEARS)} yearly tasks submitted")

print(f"\n{len(tasks)} total tasks running in parallel on GEE.")

## Step 5 — Monitor Task Status (re-run until all COMPLETED)

In [ ]:
if not tasks:
    print("No tasks to monitor.")
else:
    rows = []
    for (slug, year), task in tasks.items():
        s = task.status()
        rows.append({'city': city_bounds[slug]['label'], 'year': year,
                     'state': s['state'], 'error': s.get('error_message', '')})
    df = pd.DataFrame(rows)

    # Summary by city
    summary = (df.groupby('city')['state']
                 .value_counts().unstack(fill_value=0)
                 .reindex(columns=['COMPLETED', 'RUNNING', 'READY', 'FAILED'], fill_value=0))
    print(summary.to_string())

    n_done   = (df['state'] == 'COMPLETED').sum()
    n_failed = (df['state'] == 'FAILED').sum()
    n_pend   = df['state'].isin(['READY', 'RUNNING']).sum()
    print(f"\nTotal — Completed: {n_done}  |  Failed: {n_failed}  |  Pending: {n_pend} / {len(tasks)}")
    if n_failed:
        print("\nFailed tasks:")
        print(df[df['state'] == 'FAILED'][['city', 'year', 'error']].to_string(index=False))

## Step 6 — Download from Drive & Convert to NetCDF

Run once all tasks show COMPLETED.

In [ ]:
import json as _json, tempfile, re, requests
import rasterio
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from ee.oauth import CLIENT_ID, CLIENT_SECRET

_cred_file = Path.home() / '.config' / 'earthengine' / 'credentials'
with open(_cred_file) as _f:
    _c = _json.load(_f)

_creds = Credentials(
    token=None, refresh_token=_c['refresh_token'],
    token_uri='https://oauth2.googleapis.com/token',
    client_id=CLIENT_ID, client_secret=CLIENT_SECRET,
    scopes=['https://www.googleapis.com/auth/drive'],
)
_creds.refresh(Request())
drive_svc = build('drive', 'v3', credentials=_creds)

folder_q = f"name='{GDRIVE_FOLDER}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
folders = drive_svc.files().list(q=folder_q, fields='files(id,name)').execute().get('files', [])
if not folders:
    raise RuntimeError(f"Drive folder '{GDRIVE_FOLDER}' not found")
folder_id = folders[0]['id']
print(f"Drive folder: {GDRIVE_FOLDER} ({folder_id})\n")


def download_file(file_id, dest_path):
    if _creds.expired:
        _creds.refresh(Request())
    url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media"
    headers = {'Authorization': f'Bearer {_creds.token}'}
    with requests.get(url, headers=headers, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)


def yearly_tifs_to_netcdf(tif_paths, out_path):
    yearly_das = []
    for tp in sorted(tif_paths):
        with rasterio.open(tp) as src:
            descriptions = src.descriptions
        dates = pd.to_datetime(
            [re.search(r'(\d{8})', d).group(1) for d in descriptions],
            format='%Y%m%d'
        )
        ds = xr.open_dataset(tp, engine='rasterio')
        da = (ds['band_data']
              .assign_coords(band=dates)
              .rename({'band': 'time', 'x': 'lon', 'y': 'lat'}))
        yearly_das.append(da)

    combined = xr.concat(yearly_das, dim='time')
    combined.name = 'tmin'
    combined.attrs = {'units': 'degC', 'source': 'GSHTD / Zhang et al. (2022)'}
    combined.attrs.pop('grid_mapping', None)
    combined.to_netcdf(out_path)


for slug, bounds in city_bounds.items():
    out_path = PIPELINE_DATA / f'{slug}_tmin.nc'
    label    = bounds['label']

    if out_path.exists():
        print(f"  [{label}] already exists — skipping")
        continue

    year_tasks = {yr: tasks[(slug, yr)] for yr in YEARS if (slug, yr) in tasks}
    not_done   = [yr for yr, t in year_tasks.items() if t.status()['state'] != 'COMPLETED']
    if not_done:
        print(f"  [{label}] {len(not_done)} years not yet COMPLETED — skipping")
        continue

    print(f"  [{label}] downloading {len(YEARS)} yearly files...", flush=True)
    with tempfile.TemporaryDirectory() as tmpdir:
        tif_paths = []
        for year in sorted(YEARS):
            q = f"'{folder_id}' in parents and name='{slug}_tmin_{year}.tif' and trashed=false"
            files = drive_svc.files().list(q=q, fields='files(id,name)').execute().get('files', [])
            if not files:
                print(f"    {year}: file not found in Drive")
                break
            dest = Path(tmpdir) / f'{slug}_tmin_{year}.tif'
            download_file(files[0]['id'], str(dest))
            print(f"    {year} ✓", flush=True)
            tif_paths.append(str(dest))
        else:
            yearly_tifs_to_netcdf(tif_paths, out_path)
            da = xr.open_dataset(out_path)['tmin']
            print(f"    saved → {out_path}  ({da.sizes['time']} days, "
                  f"range {float(da.min()):.1f}–{float(da.max()):.1f} °C)")

## Step 7 — Validate

In [ ]:
rows = []
for slug, bounds in city_bounds.items():
    path = PIPELINE_DATA / f'{slug}_tmin.nc'
    if not path.exists():
        rows.append({'city': bounds['label'], 'status': 'MISSING', 'days': None, 'min_C': None, 'max_C': None})
        continue
    da = xr.open_dataset(path)['tmin']
    rows.append({'city': bounds['label'], 'status': 'OK',
                 'days': da.sizes['time'],
                 'min_C': round(float(da.min()), 1),
                 'max_C': round(float(da.max()), 1)})
print(pd.DataFrame(rows).to_string(index=False))